## Read data from team statistics

This handles data from s3 bucket and loads it to a bronze layer table cant be consumed using delta live tables
so im using autoloader

In [ ]:
from pyspark.sql.functions import current_timestamp

pass parameters to bundle later

In [ ]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
source_path = dbutils.widgets.get("source_path")
file_format = "json"

In [ ]:
def ingest_teams_statistics():
    df = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", file_format)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", f"{source_path}/schemas/teams_statistics/") #auto schema inference location
        .load(f"{source_path}/teams_statistics/")
        .withColumn("ingestion_timestamp", current_timestamp())
    )
    return df

In [ ]:
def load_players_statistics(df) -> None:
    query = (
        df.writeStream.format("delta")
        .option("checkpointLocation", f"/tmp/checkpoints/teams_statistics/")
        .outputMode("append")
        .trigger(availableNow=True)
        .table(f'{catalog}.{schema}.teams_statistics')
    )
    query.awaitTermination()

In [ ]:
df = ingest_teams_statistics()
load_players_statistics(df)